In [ ]:
!pip install rioxarray

In [ ]:
import geopandas as gpd
import rioxarray
import rioxarray.merge
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("All libraries imported successfully.")

In [ ]:
# --- 1. Configuration ---

BASE_DATA_PATH = Path(r"/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second")

# Define sub-folder names
LULC_DIR_NAME = "LULC_Annual"
RIVER_DIR_NAME = "Rivers2"
RESERVOIR_DIR_NAME = "Reservoirs2"

# --- Define the analysis distance (e.g., 5 kilometers)
BUFFER_KM = 5
BUFFER_METERS = BUFFER_KM * 1000

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !! LULC DICTIONARY - VERIFIED & CORRECTED                           !!
# !! Based on official Bhuvan LULC 250K legends.                      !!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
LULC_CLASSES = {
    0: "No Data / Background",
    31: "Built-Up",
    43: "Kharif Crop",
    82: "Rabi Crop",
    94: "Zaid Crop",
    107: "Double / Triple / Annual Crop",
    115: "Current Fallow",
    120: "Plantation / Orchard",
    133: "Evergreen / Semi-Evergreen Forest",
    158: "Deciduous Forest",
    181: "Degraded Forest / Scrub",
    184: "Littoral / Swamp / Mangroves",
    204: "Grassland",
    207: "Shifting Cultivation",
    209: "Wasteland",
    219: "Rann",
    222: "Waterbodies (Reservoir, Lake, etc.)",
    242: "Snow cover / Glacial areas",
    245: "Barren / Unculturable",
    255: "No Data / Background"
}
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# --- End of Configuration ---

print(f"Base path set to: {BASE_DATA_PATH}")
print(f"LULC Legend contains {len(LULC_CLASSES)} classes.")

In [ ]:
def analyze_lulc_and_water_area(base_path, lulc_dir, river_dir, reservoir_dir):
    """
    Performs a time-series analysis of LULC within a buffer
    around seasonal water bodies and calculates the area of the water bodies.

    Returns:
        pd.DataFrame: DataFrame with LULC percentages per year.
        pd.DataFrame: DataFrame with water body areas (km²) per year.
    """

    # Suppress ignorable warnings
    warnings.filterwarnings('ignore', category=FutureWarning)
    warnings.filterwarnings('ignore', category=UserWarning)

    # Define full paths
    lulc_path = base_path / lulc_dir
    river_path = base_path / river_dir
    reservoir_path = base_path / reservoir_dir

    if not base_path.exists():
        print(f"[ERROR] Base path not found: {base_path}")
        return None, None

    print(f"Starting analysis for LULC within {BUFFER_KM}km of water bodies...")

    # Find all the annual LULC raster files
    lulc_files = sorted(lulc_path.glob("LULC_*.tif"))
    if not lulc_files:
        print(f"[ERROR] No LULC .tif files found in: {lulc_path}")
        return None, None

    print(f"Found {len(lulc_files)} LULC files to process.")

    all_lulc_results = {}
    all_water_stats = {}

    PROJECTED_CRS_FOR_AREA = "EPSG:32644" # WGS 84 / UTM Zone 44N

    # --- 2. Loop Through Each LULC File ---
    for lulc_file in lulc_files:
        try:
            print(f"\n--- Processing: {lulc_file.name} ---")

            # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
            # !! --- NEW LOGIC: Parse Causal Year ---          !!
            # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
            year_str = lulc_file.stem.split('_')[1] # e.g., "1718", "1819", "2021"

            if len(year_str) == 4 and year_str.startswith("1"): # Handles "1718", "1819"
                causal_year_short = year_str[0:2] # e.g., "17" from "1718"
                analysis_year = f"20{causal_year_short}" # e.g., "2017"

            elif len(year_str) == 4 and year_str.startswith("2"): # Handles "2021", "2122", "2223"
                if year_str == "2021":
                    # LULC_2021.tif. Assume 2020-21, so causal monsoon is 2020.
                    analysis_year = "2020"
                else:
                    # Handles "2122", "2223", "2324"
                    causal_year_short = year_str[0:2] # e.g., "21" from "2122"
                    analysis_year = f"20{causal_year_short}" # e.g., "2021"
            else:
                print(f"  [WARNING] Skipping {lulc_file.name}: Unrecognized year format '{year_str}'.")
                continue

            print(f"  Parsed causal year: {analysis_year} (from LULC file {lulc_file.name})")
            # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
            # !! --- END OF NEW LOGIC ---                      !!
            # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

            # --- 3. Find Corresponding MONSOON Water Files ---
            river_shp_pattern = f"rivers_{analysis_year}_03_Monsoon*.shp"
            reservoir_shp_pattern = f"reservoirs_{analysis_year}_03_Monsoon*.shp"

            river_shp_list = list(river_path.glob(river_shp_pattern))
            reservoir_shp_list = list(reservoir_path.glob(reservoir_shp_pattern))

            if not river_shp_list or not reservoir_shp_list:
                print(f"  [WARNING] Could not find matching MONSOON shapefiles for {analysis_year}. Skipping this year.")
                continue

            river_shp_file = river_shp_list[0]
            reservoir_shp_file = reservoir_shp_list[0]

            print(f"  Found matching river: {river_shp_file.name}")
            print(f"  Found matching reservoir: {reservoir_shp_file.name}")

            # --- 4. Load Data and Calculate Area ---
            lulc_raster = rioxarray.open_rasterio(lulc_file, masked=True).squeeze()
            rivers_gdf = gpd.read_file(river_shp_file)
            reservoirs_gdf = gpd.read_file(reservoir_shp_file)

            # 1. Reproject a COPY to a METRIC CRS (UTM)
            rivers_gdf_proj = rivers_gdf.to_crs(PROJECTED_CRS_FOR_AREA)
            reservoirs_gdf_proj = reservoirs_gdf.to_crs(PROJECTED_CRS_FOR_AREA)

            # 2. Calculate area in square meters
            river_area_sq_meters = rivers_gdf_proj.area.sum()
            reservoir_area_sq_meters = reservoirs_gdf_proj.area.sum()

            # 3. Convert from m² to km²
            river_area_km2 = river_area_sq_meters / 1_000_000
            reservoir_area_km2 = reservoir_area_sq_meters / 1_000_000

            print(f"  Monsoon River Area: {river_area_km2:.2f} km²")
            print(f"  Monsoon Reservoir Area: {reservoir_area_km2:.2f} km²")

            all_water_stats[analysis_year] = {
                'River_Area_km2': river_area_km2,
                'Reservoir_Area_km2': reservoir_area_km2
            }

            # --- 5. Continue with LULC Buffer Analysis ---
            rivers_gdf_clip = rivers_gdf.to_crs(lulc_raster.rio.crs)
            reservoirs_gdf_clip = reservoirs_gdf.to_crs(lulc_raster.rio.crs)

            all_water_bodies = gpd.pd.concat([rivers_gdf_clip, reservoirs_gdf_clip])
            all_water_bodies_unified = all_water_bodies.unary_union

            print(f"  Creating {BUFFER_METERS}m buffer zone...")
            buffer_geom = all_water_bodies_unified.buffer(BUFFER_METERS)
            buffer_gdf = gpd.GeoDataFrame(geometry=[buffer_geom], crs=lulc_raster.rio.crs)

            print("  Clipping LULC raster to buffer...")
            lulc_clipped = lulc_raster.rio.clip(buffer_gdf.geometry, all_touched=True)

            # --- 6. Analyze Patterns & Store Results ---
            clipped_data = lulc_clipped.values
            unique_classes, pixel_counts = np.unique(clipped_data[~np.isnan(clipped_data)], return_counts=True)
            total_pixels = np.sum(pixel_counts)
            print(f"  Total pixels in buffer: {total_pixels:,}")

            year_results = {}
            for class_name in LULC_CLASSES.values():
                year_results[class_name] = 0.0 # Initialize all to 0

            for class_value, count in zip(unique_classes, pixel_counts):
                class_name = LULC_CLASSES.get(class_value, f"UNKNOWN_CLASS_{class_value}")
                percentage = (count / total_pixels) * 100
                if "UNKNOWN" in class_name:
                     print(f"  [!] Found {class_name} - Check LULC_CLASSES dictionary.")
                else:
                    year_results[class_name] += percentage

            all_lulc_results[analysis_year] = year_results
            print(f"  Analysis for {analysis_year} complete.")

        except Exception as e:
            print(f"[ERROR] Failed to process {lulc_file.name}. Error: {e}")
            continue

    # --- 7. Report Final Time-Series Table ---
    if not all_lulc_results:
        print("\nNo results were generated. Please check your file paths and names.")
        return None, None

    print("\n--- SCRIPT FINISHED: TIME-SERIES LULC ANALYSIS ---")

    lulc_df = pd.DataFrame.from_dict(all_lulc_results, orient='index')
    lulc_df = lulc_df.fillna(0.0)
    lulc_df.index.name = "Year"

    water_stats_df = pd.DataFrame.from_dict(all_water_stats, orient='index')
    water_stats_df.index.name = "Year"

    ordered_names = list(dict.fromkeys(LULC_CLASSES.values()))
    final_ordered_columns = [name for name in ordered_names if name in lulc_df.columns]
    lulc_df = lulc_df[final_ordered_columns]

    print(f"\nPercentage (%) of LULC class within the {BUFFER_KM}km Monsoon Buffer:")
    print(lulc_df.to_string(float_format="%.2f%%"))

    csv_output_path = base_path / "lulc_buffer_analysis_results.csv"
    lulc_df.to_csv(csv_output_path, float_format="%.4f")
    print(f"\nLULC results saved to: {csv_output_path}")

    csv_water_path = base_path / "water_area_results.csv"
    water_stats_df.to_csv(csv_water_path, float_format="%.4f")
    print(f"Water area results saved to: {csv_water_path}")

    return lulc_df, water_stats_df

print("Analysis function (Cell 3) has been updated and redefined.")

In [ ]:
# --- Run the Analysis ---
lulc_df, water_df = analyze_lulc_and_water_area(
    base_path=BASE_DATA_PATH,
    lulc_dir=LULC_DIR_NAME,
    river_dir=RIVER_DIR_NAME,
    reservoir_dir=RESERVOIR_DIR_NAME
)

# --- Prepare Data for Plotting ---
if lulc_df is not None and water_df is not None:
    # Combine the two dataframes
    plot_df = lulc_df.merge(water_df, left_index=True, right_index=True)

    # --- Convert index to integer for correct plot spacing ---
    plot_df.index = plot_df.index.astype(int)
    plot_df = plot_df.sort_index() # Ensure years are in order
    # --- END OF UPDATE ---

    # Group all Agriculture classes
    agri_classes = [
        "Kharif Crop", "Rabi Crop", "Zaid Crop",
        "Double / Triple / Annual Crop", "Current Fallow"
    ]
    agri_cols_to_sum = [col for col in agri_classes if col in plot_df.columns]
    plot_df["Agriculture (Total)"] = plot_df[agri_cols_to_sum].sum(axis=1)

    # Group all Vegetation/Forest classes
    veg_classes = [
        "Plantation / Orchard", "Evergreen / Semi-Evergreen Forest",
        "Deciduous Forest", "Degraded Forest / Scrub", "Grassland"
    ]
    veg_cols_to_sum = [col for col in veg_classes if col in plot_df.columns]
    plot_df["Vegetation (Total)"] = plot_df[veg_cols_to_sum].sum(axis=1)

    # Group Wasteland/Barren
    waste_classes = ["Wasteland", "Barren / Unculturable", "Rann"]
    waste_cols_to_sum = [col for col in waste_classes if col in plot_df.columns]
    plot_df["Wasteland/Barren (Total)"] = plot_df[waste_cols_to_sum].sum(axis=1)

    # Select the key columns for our plots
    key_plot_cols = [
        'Built-Up',
        'Agriculture (Total)',
        'Vegetation (Total)',
        'Wasteland/Barren (Total)',
        'Waterbodies (Reservoir, Lake, etc.)',
        'River_Area_km2',
        'Reservoir_Area_km2'
    ]

    final_plot_df = plot_df[[col for col in key_plot_cols if col in plot_df.columns]]

    print("\n--- Data Prepared for Plotting ---")
    print(final_plot_df.to_string(float_format="%.2f"))
else:
    print("Analysis failed. No data to plot.")

PLOTS

In [ ]:
if 'final_plot_df' in locals():
    print("Generating Plot 1: Reservoir Area vs. LULC Dashboard...")

    fig, ax1 = plt.subplots(figsize=(14, 8))

    # --- Left Y-Axis (LULC Percentages) ---
    # --- UPDATED: Color map with all 4 categories ---
    color_map = {
        'Built-Up': 'red',
        'Agriculture (Total)': 'orange',
        'Vegetation (Total)': 'green',
        'Wasteland/Barren (Total)': 'gray'
    }

    ax1.set_xlabel('Year', fontsize=12)
    ax1.set_ylabel('LULC Change (% of 5km Buffer)', fontsize=12, color='black')
    ax1.tick_params(axis='y', labelcolor='black')

    # --- UPDATED: Loop to plot all 4 LULC lines ---
    for col, color in color_map.items():
        if col in final_plot_df.columns:
            ax1.plot(final_plot_df.index, final_plot_df[col],
                     label=f'LULC: {col}', color=color, marker='o', linestyle='-')

    ax1.yaxis.set_major_formatter(mticker.PercentFormatter())

    # --- Right Y-Axis (Reservoir Area) ---
    ax2 = ax1.twinx()
    ax2.set_ylabel('Monsoon Reservoir Area (km²)', fontsize=12, color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    if 'Reservoir_Area_km2' in final_plot_df.columns:
        ax2.plot(final_plot_df.index, final_plot_df['Reservoir_Area_km2'],
                 label='Water: Reservoir Area (km²)', color='blue', marker='s', linestyle='--')

    # --- Final Touches ---
    fig.suptitle('Reservoir Area vs. Key LULC Changes in 5km Buffer', fontsize=16, y=1.02)
    fig.tight_layout()
    ax1.set_xticks(final_plot_df.index)

    # Combine legends
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper left', bbox_to_anchor=(0, -0.1), ncol=3, fancybox=True, shadow=True)

    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()

else:
    print("No data to plot. Run previous cells (1-4) first.")

In [ ]:
if 'final_plot_df' in locals():
    print("Generating Plot 2: River Surface Area vs. LULC Dashboard...")

    fig, ax1 = plt.subplots(figsize=(14, 8))

    # --- Left Y-Axis (LULC Percentages) ---
    # --- UPDATED: Color map with all 4 categories ---
    color_map = {
        'Built-Up': 'red',
        'Agriculture (Total)': 'orange',
        'Vegetation (Total)': 'green',
        'Wasteland/Barren (Total)': 'gray'
    }

    ax1.set_xlabel('Year', fontsize=12)
    ax1.set_ylabel('LULC Change (% of 5km Buffer)', fontsize=12, color='black')
    ax1.tick_params(axis='y', labelcolor='black')

    # --- UPDATED: Loop to plot all 4 LULC lines ---
    for col, color in color_map.items():
        if col in final_plot_df.columns:
            ax1.plot(final_plot_df.index, final_plot_df[col],
                     label=f'LULC: {col}', color=color, marker='o', linestyle='-')

    ax1.yaxis.set_major_formatter(mticker.PercentFormatter())

    # --- Right Y-Axis (River Area) ---
    ax2 = ax1.twinx()
    ax2.set_ylabel('Monsoon River Surface Area (km²)', fontsize=12, color='purple')
    ax2.tick_params(axis='y', labelcolor='purple')

    if 'River_Area_km2' in final_plot_df.columns:
        ax2.plot(final_plot_df.index, final_plot_df['River_Area_km2'],
                 label='Water: River Area (km²)', color='purple', marker='X', linestyle=':')

    # --- Final Touches ---
    fig.suptitle('River Surface Area vs. Key LULC Changes in 5km Buffer', fontsize=16, y=1.02)
    fig.tight_layout()
    ax1.set_xticks(final_plot_df.index)

    # Combine legends
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper left', bbox_to_anchor=(0, -0.1), ncol=3, fancybox=True, shadow=True)

    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()

else:
    print("No data to plot. Run previous cells (1-4) first.")

In [ ]:
if 'final_plot_df' in locals():
    print("Generating Plot 3: Correlation Scatter Plots...")

    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16, 7))

    # Define colors for legend
    color_map = {
        'Built-Up': 'red',
        'Agriculture (Total)': 'orange',
        'Vegetation (Total)': 'green',
        'Wasteland/Barren (Total)': 'gray'
    }

    # --- Plot 1: Reservoir Area vs. LULC ---
    for col, color in color_map.items():
        if col in final_plot_df.columns:
            ax1.scatter(final_plot_df['Reservoir_Area_km2'], final_plot_df[col],
                        color=color, label=col, s=80, alpha=0.7)

    ax1.set_title('Reservoir Area vs. LULC in Buffer', fontsize=14, pad=10)
    ax1.set_xlabel('Monsoon Reservoir Area (km²)', fontsize=12)
    ax1.set_ylabel('LULC Class (% of 5km Buffer)', fontsize=12)
    ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax1.legend()
    ax1.grid(True, linestyle=':', alpha=0.6)

    # --- Plot 2: River Area vs. LULC ---
    for col, color in color_map.items():
        if col in final_plot_df.columns:
            ax2.scatter(final_plot_df['River_Area_km2'], final_plot_df[col],
                        color=color, label=col, s=80, alpha=0.7)

    ax2.set_title('River Surface Area vs. LULC in Buffer', fontsize=14, pad=10)
    ax2.set_xlabel('Monsoon River Surface Area (km²)', fontsize=12)
    ax2.set_ylabel('LULC Class (% of 5km Buffer)', fontsize=12)
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax2.legend()
    ax2.grid(True, linestyle=':', alpha=0.6)

    fig.suptitle('LULC Correlation with Water Body Size', fontsize=18, y=1.03)
    plt.tight_layout(pad=2.0)
    plt.show()

else:
    print("No data to plot. Run previous cells (1-4) first.")

In [ ]:
if 'final_plot_df' in locals():
    print("Generating Plot 4: LULC Composition (Stacked Area Chart)...")

    lulc_cols_to_plot = [
        'Built-Up',
        'Agriculture (Total)',
        'Vegetation (Total)',
        'Wasteland/Barren (Total)',
        'Waterbodies (Reservoir, Lake, etc.)'
    ]

    lulc_data_to_stack = final_plot_df[[col for col in lulc_cols_to_plot if col in final_plot_df]]

    # --- Create the 100% Stacked Chart ---
    lulc_percent = lulc_data_to_stack.divide(lulc_data_to_stack.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(14, 8))

    color_map = {
        'Built-Up': 'red',
        'Agriculture (Total)': 'orange',
        'Vegetation (Total)': 'green',
        'Wasteland/Barren (Total)': 'gray',
        'Waterbodies (Reservoir, Lake, etc.)': 'blue'
    }

    labels = lulc_percent.columns
    colors = [color_map.get(label, 'black') for label in labels]

    ax.stackplot(lulc_percent.index, lulc_percent.T, labels=labels, colors=colors, alpha=0.8)

    # --- Final Touches ---
    ax.set_title('Yearly LULC Composition in 5km Water Body Buffer', fontsize=16, pad=15)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Percentage Share of LULC (%)', fontsize=12)

    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_xticks(lulc_percent.index)

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), title="LULC Classes", fancybox=True, shadow=True)

    plt.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()

else:
    print("No data to plot. Run previous cells (1-4) first.")

In [ ]:
if 'final_plot_df' in locals():
    print("Generating Plot 5: Net LULC Change Analysis (2018 vs. Last Year)...")

    try:
        # --- Calculate Net Change ---
        # Get the first and last year's data
        first_year = final_plot_df.index.min()
        last_year = final_plot_df.index.max()

        start_data = final_plot_df.loc[first_year]
        end_data = final_plot_df.loc[last_year]

        # Calculate the change in percentage points
        net_change = end_data - start_data

        # Select just the grouped LULC columns for this plot
        lulc_cols = [
            'Built-Up',
            'Agriculture (Total)',
            'Vegetation (Total)',
            'Wasteland/Barren (Total)',
            'Waterbodies (Reservoir, Lake, etc.)'
        ]

        net_change_lulc = net_change.filter(items=lulc_cols)
        net_change_lulc = net_change_lulc.sort_values(ascending=False)

        # --- Create the Bar Plot ---
        fig, ax = plt.subplots(figsize=(12, 7))

        # Create a color list: green for positive, red for negative
        colors = ['g' if x > 0 else 'r' for x in net_change_lulc]

        net_change_lulc.plot(kind='bar', ax=ax, color=colors, zorder=3)

        # --- Final Touches ---
        ax.set_title(f'Net LULC Change in 5km Buffer ({first_year} vs. {last_year})', fontsize=16, pad=15)
        ax.set_xlabel('LULC Class', fontsize=12)
        ax.set_ylabel('Change in Percentage Points (p.p.)', fontsize=12)

        # Add horizontal line at y=0
        ax.axhline(0, color='black', linewidth=0.8, zorder=2)

        # Add labels on top of bars
        for p in ax.patches:
            ax.annotate(f'{p.get_height():+.2f} p.p.',
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center',
                        xytext=(0, 10 if p.get_height() > 0 else -10),
                        textcoords='offset points')

        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.grid(axis='y', linestyle=':', alpha=0.7, zorder=0)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Could not generate Net Change plot. Error: {e}")
        print("This often happens if there is only one year of data.")

else:
    print("No data to plot. Run previous cells (1-4) first.")